# 01 — Data audit and sampling

**Question answered:** Q0 in `docs/01_analysis_plan.md` — is the data usable, and is my sample representative?

**Gate (pre-committed, do not move):**

| Check | Tolerance |
|---|---|
| Sample churn rate vs population | 0.5 pp |
| Auto-renew share vs population | 2.0 pp |
| Plan-length mix (top 5) vs population | 2.0 pp |

**Outputs:** a data quality table, `data/sample_msno.txt`, and entries for `docs/02_decision_log.md`.

**Do not proceed to the SQLite build until the gate passes.**


## 0. Setup


In [1]:
from pathlib import Path
import subprocess

import numpy as np
import pandas as pd

PROJECT = Path.cwd().parent          # this notebook lives in notebooks/
RAW  = PROJECT / 'data' / 'raw'      # .7z archives, never committed
WORK = PROJECT / 'data'              # extracted + derived files

SEED = 42
SAMPLE_N = 200_000

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)

print('project root :', PROJECT)
print('archives     :', RAW, '->', len(list(RAW.glob('*.7z'))), 'files')


/Users/laxmigupte/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


project root : /Users/laxmigupte/Desktop/Subscriber-Retention-Budget-Allocation-
archives     : /Users/laxmigupte/Desktop/Subscriber-Retention-Budget-Allocation-/data/raw -> 6 files


## 1. Extract the small archives

`user_logs.csv.7z` is deliberately excluded — it is streamed from the terminal at the end of this notebook, never extracted.


In [5]:
import shutil
import subprocess
import tempfile
from pathlib import Path

SMALL = {
    'members_v3.csv.7z':      'members_v3.csv',
    'train_v2.csv.7z':        'train_v2.csv',
    'transactions.csv.7z':    'transactions.csv',
    'transactions_v2.csv.7z': 'transactions_v2.csv',
    'user_logs_v2.csv.7z':    'user_logs_v2.csv',
}


def inner_names(archive):
    listing = subprocess.run(
        ['7z', 'l', '-ba', str(RAW / archive)],
        capture_output=True, text=True, check=True).stdout
    return [ln.split()[-1] for ln in listing.splitlines() if ln.strip()]


def extract_one(archive, target_name):
    target = WORK / target_name
    if target.exists():
        print(f'already present : {target_name:<22} {target.stat().st_size / 1e9:.2f} GB')
        return target

    tmp = Path(tempfile.mkdtemp(dir=WORK, prefix='_unpack_'))
    try:
        subprocess.run(['7z', 'x', '-y', f'-o{tmp}', str(RAW / archive)],
                       check=True, capture_output=True, text=True)

        produced = [p for p in tmp.rglob('*') if p.is_file()]
        if len(produced) != 1:
            raise RuntimeError(f'{archive}: expected one file, got {[p.name for p in produced]}')

        inner = produced[0]
        if inner.parent != tmp:
            print(f'  note: archive nested it under "{inner.parent.relative_to(tmp)}/", flattening')
        shutil.move(str(inner), str(target))
    finally:
        shutil.rmtree(tmp, ignore_errors=True)

    print(f'extracted       : {target_name:<22} {target.stat().st_size / 1e9:.2f} GB')
    return target


print('--- what is actually inside each archive ---')
for a in sorted(SMALL):
    print(f'{a:<26} contains {inner_names(a)}')

print('\n--- extracting ---')
for archive, name in SMALL.items():
    extract_one(archive, name)

print('\n--- header check ---')
for name in SMALL.values():
    with open(WORK / name) as f:
        print(f'{name:<22} {f.readline().strip()[:90]}')

--- what is actually inside each archive ---
members_v3.csv.7z          contains ['members_v3.csv']
train_v2.csv.7z            contains ['data/churn_comp_refresh/train_v2.csv']
transactions.csv.7z        contains ['transactions.csv']
transactions_v2.csv.7z     contains ['data/churn_comp_refresh/transactions_v2.csv']
user_logs_v2.csv.7z        contains ['data/churn_comp_refresh/user_logs_v2.csv']

--- extracting ---
already present : members_v3.csv         0.43 GB
  note: archive nested it under "data/churn_comp_refresh/", flattening
extracted       : train_v2.csv           0.05 GB
extracted       : transactions.csv       1.73 GB
  note: archive nested it under "data/churn_comp_refresh/", flattening
extracted       : transactions_v2.csv    0.12 GB
  note: archive nested it under "data/churn_comp_refresh/", flattening
extracted       : user_logs_v2.csv       1.43 GB

--- header check ---
members_v3.csv         msno,city,bd,gender,registered_via,registration_init_time
train_v2.csv        

## 2. Load the label set and member profiles


In [6]:
labels  = pd.read_csv(WORK / 'train_v2.csv')
members = pd.read_csv(WORK / 'members_v3.csv')

print(f'labels  : {labels.shape[0]:,} rows x {labels.shape[1]} cols')
print(f'members : {members.shape[0]:,} rows x {members.shape[1]} cols')
labels.head()


labels  : 970,960 rows x 2 cols
members : 6,769,473 rows x 6 cols


,msno,is_churn
0,ugx0CjOMzazClkFzU2xasmDZaoIqOUAZPsH1q0teWCg=,1
1,f/NmvEzHfhINFEYZTR05prUdr+E+3+oewvweYz9cCQE=,1
2,zLo9f73nGGT1p21ltZC3ChiRnAVvgibMyazbCxvWPcg=,1
3,8iF/+8HY8lJKFrTc7iR9ZYGCG2Ecrogbc2Vy5YhsfhQ=,1
4,K6fja4+jmoZ5xG6BypqX80Uw/XKpMgrEMdG2edFOxnA=,1


In [7]:
pop_churn = labels['is_churn'].mean()
print(f'POPULATION CHURN RATE: {pop_churn:.4%}  (n = {len(labels):,})')
print('This is the number the sample must reproduce within 0.5 pp.')


POPULATION CHURN RATE: 8.9942%  (n = 970,960)
This is the number the sample must reproduce within 0.5 pp.


## 3. Data quality profile

Every anomaly found here gets a stated handling rule. Nothing is fixed silently.


In [8]:
def profile(df, name):
    out = pd.DataFrame({
        'dtype':    df.dtypes.astype(str),
        'nulls':    df.isna().sum(),
        'null_pct': (df.isna().mean() * 100).round(2),
        'n_unique': df.nunique(),
    })
    out.index.name = name
    return out

profile(members, 'members_v3')


,dtype,nulls,null_pct,n_unique
members_v3,,,,
msno,object,0,0.00,6769473
city,int64,0,0.00,21
bd,int64,0,0.00,386
gender,object,4429505,65.43,2
registered_via,int64,0,0.00,18
registration_init_time,int64,0,0.00,4782


In [9]:
tx = pd.read_csv(WORK / 'transactions_v2.csv')
print(f'transactions_v2 : {tx.shape[0]:,} rows')
profile(tx, 'transactions_v2')


transactions_v2 : 1,431,009 rows


,dtype,nulls,null_pct,n_unique
transactions_v2,,,,
msno,object,0,0.0,1197050
payment_method_id,int64,0,0.0,37
payment_plan_days,int64,0,0.0,31
plan_list_price,int64,0,0.0,48
actual_amount_paid,int64,0,0.0,53
is_auto_renew,int64,0,0.0,2
transaction_date,int64,0,0.0,820
membership_expire_date,int64,0,0.0,1960
is_cancel,int64,0,0.0,2


### 3a. Age (`bd`) outliers

This field is self-reported and notoriously dirty. Decide the rule here, apply it later.


In [10]:
bd = members['bd']
print(f'negative          : {(bd < 0).sum():,}')
print(f'zero (unstated)   : {(bd == 0).sum():,}')
print(f'implausibly young : {((bd > 0) & (bd < 13)).sum():,}')
print(f'implausibly old   : {(bd > 100).sum():,}')
print(f'plausible (13-100): {((bd >= 13) & (bd <= 100)).sum():,}')
print()
print(f'gender missing    : {members["gender"].isna().mean():.1%}')


negative          : 274
zero (unstated)   : 4,540,215
implausibly young : 1,301
implausibly old   : 5,377
plausible (13-100): 2,222,306

gender missing    : 65.4%


**Handling rule (log this):** values outside 13–100 are set to null and flagged with an `age_stated` boolean, rather than dropped. 
Dropping the rows would discard otherwise-valid subscribers; the flag lets the model use *whether* age was stated, which is itself informative.


### 3b. Date parsing

All dates arrive as integers in `YYYYMMDD` form.


In [11]:
for col in ['transaction_date', 'membership_expire_date']:
    tx[col] = pd.to_datetime(tx[col], format='%Y%m%d', errors='coerce')

members['registration_init_time'] = pd.to_datetime(
    members['registration_init_time'], format='%Y%m%d', errors='coerce')

print('transaction_date       :', tx['transaction_date'].min().date(), '->', tx['transaction_date'].max().date())
print('membership_expire_date :', tx['membership_expire_date'].min().date(), '->', tx['membership_expire_date'].max().date())
print('unparseable dates      :', tx[['transaction_date', 'membership_expire_date']].isna().sum().sum())


transaction_date       : 2015-01-01 -> 2017-03-31
membership_expire_date : 2016-04-19 -> 2036-10-15
unparseable dates      : 0


### 3c. Duplicate transactions


In [12]:
dup = tx.duplicated(subset=['msno', 'transaction_date']).sum()
print(f'duplicate (msno, transaction_date) pairs: {dup:,}  ({dup / len(tx):.2%})')
print('Deduplication happens in sql/01_clean_transactions.sql, not here.')


duplicate (msno, transaction_date) pairs: 33,292  (2.33%)
Deduplication happens in sql/01_clean_transactions.sql, not here.


## 4. Provisional expiry anchor

For sampling validation only, the anchor is each subscriber's **last March transaction**. 
The authoritative anchor is rebuilt properly with a window function in `sql/01_clean_transactions.sql` — this is a shortcut for the gate check, and is labelled as such.


In [13]:
anchor = (tx.sort_values(['msno', 'transaction_date'])
            .groupby('msno', as_index=False)
            .last())

cols = ['msno', 'payment_plan_days', 'is_auto_renew', 'is_cancel',
        'plan_list_price', 'actual_amount_paid', 'payment_method_id']

pop = labels.merge(anchor[cols], on='msno', how='left')

missing = pop['payment_plan_days'].isna().sum()
print(f'cohort rows with no March transaction: {missing:,}  ({missing / len(pop):.2%})')
print('These are expected — expiry can precede any March transaction. Counted, not dropped.')


cohort rows with no March transaction: 37,382  (3.85%)
These are expected — expiry can precede any March transaction. Counted, not dropped.


## 5. Draw the sample


In [14]:
sample_ids = labels['msno'].sample(n=SAMPLE_N, random_state=SEED)
sample_set = set(sample_ids)

smp = pop[pop['msno'].isin(sample_set)]
print(f'sample: {len(smp):,} subscribers ({len(smp) / len(pop):.1%} of cohort), seed={SEED}')


sample: 200,000 subscribers (20.6% of cohort), seed=42


## 6. Validation gate

If any row fails, resample or investigate. **Do not continue on a failed gate** — and if you change the tolerance, that change goes in `docs/02_decision_log.md` with a reason.


In [15]:
rows = []

rows.append({'metric': 'churn rate %',
             'population': pop['is_churn'].mean() * 100,
             'sample':     smp['is_churn'].mean() * 100,
             'tolerance_pp': 0.5})

rows.append({'metric': 'auto-renew %',
             'population': pop['is_auto_renew'].mean() * 100,
             'sample':     smp['is_auto_renew'].mean() * 100,
             'tolerance_pp': 2.0})

top_plans = pop['payment_plan_days'].value_counts().head(5).index
for d in top_plans:
    rows.append({'metric': f'plan {int(d)}-day %',
                 'population': (pop['payment_plan_days'] == d).mean() * 100,
                 'sample':     (smp['payment_plan_days'] == d).mean() * 100,
                 'tolerance_pp': 2.0})

gate = pd.DataFrame(rows)
gate['diff_pp'] = (gate['sample'] - gate['population']).abs().round(3)
gate['pass']    = gate['diff_pp'] <= gate['tolerance_pp']
gate = gate.round({'population': 3, 'sample': 3})
gate


,metric,population,sample,tolerance_pp,diff_pp,pass
0,churn rate %,8.994,8.981,0.5,0.013,True
1,auto-renew %,91.121,91.195,2.0,0.074,True
2,plan 30-day %,94.265,94.331,2.0,0.066,True
3,plan 410-day %,0.457,0.466,2.0,0.009,True
4,plan 90-day %,0.380,0.388,2.0,0.007,True
5,plan 180-day %,0.302,0.282,2.0,0.019,True
6,plan 195-day %,0.301,0.294,2.0,0.007,True


In [16]:
assert gate['pass'].all(), (
    'SAMPLING GATE FAILED. Do not proceed. '
    'Either resample with a different seed, or investigate why the sample is skewed.'
)
print('Gate passed. Sample is representative within stated tolerances.')


Gate passed. Sample is representative within stated tolerances.


## 7. Write the subscriber list

One ID per line, no header. This file is what `src/filter_logs.py` reads.


In [17]:
out = WORK / 'sample_msno.txt'
pd.Series(sorted(sample_set)).to_csv(out, index=False, header=False)

n_lines = sum(1 for _ in open(out))
print(f'wrote {out}')
print(f'{n_lines:,} ids, {out.stat().st_size / 1e6:.1f} MB')

with open(out) as f:
    print('first line:', f.readline().strip())


wrote /Users/laxmigupte/Desktop/Subscriber-Retention-Budget-Allocation-/data/sample_msno.txt
200,000 ids, 9.0 MB
first line: ++/9R3sX37CjxbY/AaGvbwr3QkwElKBCtSvVzhCBDOk=


## 8. Decision log entries

Paste the output below into `docs/02_decision_log.md`.


In [18]:
from datetime import date

today = date.today().isoformat()

entries = f'''
- **{today}** — Sampled {SAMPLE_N:,} of {len(labels):,} labelled subscribers (seed={SEED}).
  *Rationale:* full listening history is ~30 GB; a validated sample keeps the analysis on one laptop.
  *Validation:* churn rate within {gate.loc[0, "diff_pp"]:.3f} pp of population; auto-renew and top-5 plan mix within 2 pp.
  *Cost:* reduced precision on small city and payment-method segments.

- **{today}** — Streamed `user_logs.csv.7z` through a filter rather than extracting it.
  *Rationale:* ~30 GB uncompressed, of which ~97% is subscribers outside the sample.
  *Cost:* one non-restartable 20-40 minute pass; a failure mid-run means starting over.

- **{today}** — Age (`bd`) values outside 13-100 set to null with an `age_stated` flag rather than dropped.
  *Rationale:* the rows are otherwise valid, and whether age was stated is itself a signal.
'''

print(entries)



- **2026-09-17** — Sampled 200,000 of 970,960 labelled subscribers (seed=42).
  *Rationale:* full listening history is ~30 GB; a validated sample keeps the analysis on one laptop.
  *Validation:* churn rate within 0.013 pp of population; auto-renew and top-5 plan mix within 2 pp.
  *Cost:* reduced precision on small city and payment-method segments.

- **2026-09-17** — Streamed `user_logs.csv.7z` through a filter rather than extracting it.
  *Rationale:* ~30 GB uncompressed, of which ~97% is subscribers outside the sample.
  *Cost:* one non-restartable 20-40 minute pass; a failure mid-run means starting over.

- **2026-09-17** — Age (`bd`) values outside 13-100 set to null with an `age_stated` flag rather than dropped.
  *Rationale:* the rows are otherwise valid, and whether age was stated is itself a signal.



## 9. Next step — run this in the terminal

Not in this notebook. From the **project root**, with `data/sample_msno.txt` now written:

```bash
7z e -so data/raw/user_logs.csv.7z | python3 src/filter_logs.py
```

Expect 20-40 minutes. Progress prints every 5 million rows. Leave it running and move on to the SQLite schema.

When it finishes you will have `data/logs_hist.csv` — the pre-March listening history for your sampled subscribers only.
